# KL Expressibility on IQM Spark (Hardware–Hardware Overlaps)

**Authors:** Koło Naukowe Axion

This notebook estimates **KL expressibility** on the IQM Spark quantum computer by comparing the empirical distribution of **hardware–hardware overlaps**

$$F = \mathrm{Tr}(\rho_a \rho_b)$$

against the analytic **Haar fidelity distribution** via

$$D_{\mathrm{KL}}(P_{\mathrm{hardware}}(F)\,\|\,P_{\mathrm{Haar}}(F)).$$

## Protocol

1. For a fixed ansatz $U(\theta)$ and depth, draw two independent parameter vectors $\theta_a, \theta_b \sim U[0, 2\pi)$.
2. Prepare both circuits on IQM Spark and run full $3^n$ Pauli-basis **quantum state tomography** for each state.
3. Reconstruct $\rho_a$, $\rho_b$ from measurement counts (linear inversion + PSD projection).
4. Compute $F_{\mathrm{linear}}$ and $F_{\mathrm{physical}}$; bin samples and compute $D_{\mathrm{KL}}$ vs discretized Haar.

Tomography methodology matches [`full_odra_fidelity.ipynb`](full_odra_fidelity.ipynb).

## Runtime budget

For $n=5$ qubits each fidelity sample requires **$2 \times 3^5 = 486$** tomography circuits. On IQM Spark, 243 circuits @ 1024 shots takes about **2 minutes** per state, so each pair is about **4 minutes**. With a **13-hour** budget, defaults use **30 pairs per (ansatz, depth)** across 2 ansatze and depths $\{2,4,6\}$.


## 1. Imports & Configuration

In [ ]:
import csv
import getpass
import json
import math
import os
import time
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

NUM_QUBITS = 5
DEPTHS = [2, 4, 6]
ANSATZES = {
    "ansatz_odra": None,
    "ansatz_simulator": None,
}
SEED = 42
N_SAMPLES = 30
SHOTS = 1024
MINUTES_PER_STATE_TOMOGRAPHY = 2.0
N_BINS = 150
EPS = 1e-12
DIM = 2 ** NUM_QUBITS
IQM_URL = os.environ.get("IQM_URL", "https://odra5.e-science.pl/").strip()
OPTIMIZATION_LEVEL = 1
MAX_CIRCUITS_PER_JOB = 250
NOTEBOOK_DIR = Path(".").resolve()

circuits_per_pair = 2 * (3 ** NUM_QUBITS)
total_circuits = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * circuits_per_pair
est_minutes = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * 2.0 * MINUTES_PER_STATE_TOMOGRAPHY
print(f"n={NUM_QUBITS}, circuits per fidelity pair: {circuits_per_pair}")
print(
    f"Planned run: {len(ANSATZES)} ansatze x {len(DEPTHS)} depths x "
    f"{N_SAMPLES} pairs = {total_circuits} tomography circuits x {SHOTS} shots"
)
print(f"Estimated wall time: {est_minutes:.0f} min ({est_minutes / 60:.1f} h)")


## 2. Ansatz Definitions

In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    n_macro = depth // 2
    if n_macro == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (n_macro - 1) * full + last


def ansatz_odra(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc


def ansatz_simulator(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc



ANSATZES["ansatz_odra"] = ansatz_odra
ANSATZES["ansatz_simulator"] = ansatz_simulator


## 3. Tomography, KL, and Hardware Helpers

In [ ]:
try:
    from iqm.qiskit_iqm import transpile_to_IQM as _iqm_transpile
    from iqm.qiskit_iqm.iqm_backend import IQMBackendBase as _IQMBackendBase
except ImportError:
    _iqm_transpile = None
    _IQMBackendBase = None

_PAULI_I = np.array([[1, 0], [0, 1]], dtype=complex)
_PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
_PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {"I": _PAULI_I, "X": _PAULI_X, "Y": _PAULI_Y, "Z": _PAULI_Z}


# ---------------------------------------------------------------------------
# IQM backend helpers
# ---------------------------------------------------------------------------


def connect_to_iqm_backend(iqm_url: str, token: str | None = None):
    env_token = os.environ.get("IQM_TOKEN", "").strip()
    if token and env_token:
        raise ValueError("Set either --iqm-token or IQM_TOKEN, not both")
    if token is None and not env_token:
        token = getpass.getpass("Enter IQM Token: ").strip()
    from iqm.qiskit_iqm import IQMProvider

    if token:
        provider = IQMProvider(iqm_url, token=token)
    else:
        provider = IQMProvider(iqm_url)
    return provider.get_backend()


def transpile_for_backend(
    circuit: QuantumCircuit,
    backend,
    optimization_level: int,
    seed_transpiler: int | None,
):
    kwargs: dict[str, object] = {"optimization_level": optimization_level}
    if seed_transpiler is not None:
        kwargs["seed_transpiler"] = seed_transpiler
    if (
        _iqm_transpile is not None
        and _IQMBackendBase is not None
        and isinstance(backend, _IQMBackendBase)
    ):
        return _iqm_transpile(circuit, backend, **kwargs)
    return transpile(circuit, backend, **kwargs)


def normalize_counts(counts) -> dict[str, int]:
    if isinstance(counts, list):
        if len(counts) != 1:
            raise ValueError(f"Expected one counts dict, got {len(counts)}")
        counts = counts[0]
    return {str(k): int(v) for k, v in counts.items()}


# ---------------------------------------------------------------------------
# Tomography (ported from full_odra_fidelity.ipynb)
# ---------------------------------------------------------------------------


def all_basis_settings(n_qubits: int) -> list[tuple[str, ...]]:
    return list(product(["X", "Y", "Z"], repeat=n_qubits))


def add_tomography_rotations(circuit: QuantumCircuit, bases: tuple[str, ...]) -> QuantumCircuit:
    qc = circuit.copy()
    for qubit, b in enumerate(bases):
        if b == "X":
            qc.h(qubit)
        elif b == "Y":
            qc.sdg(qubit)
            qc.h(qubit)
        elif b == "Z":
            pass
        else:
            raise ValueError(f"Unknown basis {b!r}, expected X / Y / Z")
    qc.measure_all()
    return qc


def expectation_from_counts(counts: dict[str, int], pauli_string: str) -> float:
    shots = sum(counts.values())
    if shots == 0:
        return 0.0
    expval = 0.0
    for bitstring, count in counts.items():
        bits = bitstring.replace(" ", "")[::-1]
        value = 1
        for pauli, bit in zip(pauli_string, bits):
            if pauli == "I":
                continue
            value *= 1 if bit == "0" else -1
        expval += value * count / shots
    return float(expval)


def _kron_all(ops: list[np.ndarray]) -> np.ndarray:
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out


def reconstruct_rho(tomography_counts: dict[tuple[str, ...], dict[str, int]], n_qubits: int) -> np.ndarray:
    dim = 2 ** n_qubits
    rho = np.zeros((dim, dim), dtype=complex)
    for pauli_tuple in product("IXYZ", repeat=n_qubits):
        pauli_string = "".join(pauli_tuple)
        basis_tuple = tuple(p if p != "I" else "Z" for p in pauli_tuple)
        counts = tomography_counts[basis_tuple]
        expval = expectation_from_counts(counts, pauli_string)
        P = _kron_all([PAULI[p] for p in reversed(pauli_tuple)])
        rho += expval * P
    rho /= dim
    return rho


def project_to_physical(rho: np.ndarray) -> np.ndarray:
    rho = 0.5 * (rho + rho.conj().T)
    eigvals, _ = np.linalg.eigh(rho)
    eigvals = np.sort(eigvals)[::-1]
    accum = 0.0
    n = len(eigvals)
    proj = np.zeros_like(eigvals)
    for i in range(n - 1, -1, -1):
        ev = eigvals[i] + accum / (i + 1)
        if ev >= 0:
            for j in range(i + 1):
                proj[j] = eigvals[j] + accum / (i + 1)
            break
        accum += eigvals[i]

    eigvals_unsorted, eigvecs_unsorted = np.linalg.eigh(0.5 * (rho + rho.conj().T))
    order = np.argsort(eigvals_unsorted)[::-1]
    proj_in_orig_order = np.empty_like(eigvals_unsorted)
    proj_in_orig_order[order] = proj
    return eigvecs_unsorted @ np.diag(proj_in_orig_order) @ eigvecs_unsorted.conj().T


def hardware_overlap(rho_a: np.ndarray, rho_b: np.ndarray) -> float:
    return float(np.real(np.trace(rho_a @ rho_b)))


def run_tomography_jobs(
    state_circuit: QuantumCircuit,
    backend,
    n_qubits: int,
    shots: int,
    optimization_level: int,
    seed_transpiler: int | None,
    max_circuits_per_job: int,
    label: str = "",
) -> dict[tuple[str, ...], dict[str, int]]:
    bases = all_basis_settings(n_qubits)
    tomo_circuits = [add_tomography_rotations(state_circuit, b) for b in bases]
    transpiled = [
        transpile_for_backend(qc, backend, optimization_level, seed_transpiler)
        for qc in tomo_circuits
    ]

    if label:
        print(
            f"  [{label}] submitting {len(transpiled)} tomography circuits "
            f"in batches of {max_circuits_per_job} ({shots} shots each)..."
        )

    counts_per_basis: dict[tuple[str, ...], dict[str, int]] = {}
    submitted = 0
    while submitted < len(transpiled):
        batch = transpiled[submitted : submitted + max_circuits_per_job]
        batch_bases = bases[submitted : submitted + max_circuits_per_job]
        t0 = time.perf_counter()
        result = backend.run(batch, shots=shots).result()
        dt = time.perf_counter() - t0
        counts_list = result.get_counts()
        if not isinstance(counts_list, list):
            counts_list = [counts_list]
        if len(counts_list) != len(batch):
            raise RuntimeError(
                f"Expected {len(batch)} count dicts, backend returned {len(counts_list)}"
            )
        for j, basis_tuple in enumerate(batch_bases):
            counts_per_basis[basis_tuple] = normalize_counts(counts_list[j])
        submitted += len(batch)
        if label:
            print(f"    batch done: {submitted:>4}/{len(transpiled)}  ({dt:.1f}s on backend)")

    return counts_per_basis


def tomography_density_matrices(
    state_circuit: QuantumCircuit,
    backend,
    n_qubits: int,
    shots: int,
    optimization_level: int,
    seed_transpiler: int | None,
    max_circuits_per_job: int,
    label: str = "",
) -> tuple[np.ndarray, np.ndarray, dict[str, float]]:
    counts = run_tomography_jobs(
        state_circuit,
        backend,
        n_qubits=n_qubits,
        shots=shots,
        optimization_level=optimization_level,
        seed_transpiler=seed_transpiler,
        max_circuits_per_job=max_circuits_per_job,
        label=label,
    )
    rho_lin = reconstruct_rho(counts, n_qubits)
    rho_phys = project_to_physical(rho_lin)
    diagnostics = {
        "trace_linear": float(np.real(np.trace(rho_lin))),
        "trace_physical": float(np.real(np.trace(rho_phys))),
        "purity_linear": float(np.real(np.trace(rho_lin @ rho_lin))),
        "purity_physical": float(np.real(np.trace(rho_phys @ rho_phys))),
    }
    return rho_lin, rho_phys, diagnostics


# ---------------------------------------------------------------------------
# KL / Haar utilities
# ---------------------------------------------------------------------------


def haar_pdf_fidelity(f: np.ndarray, dim: int) -> np.ndarray:
    return (dim - 1.0) * (1.0 - f) ** (dim - 2.0)


def binned_distributions(
    fid_values: np.ndarray,
    dim: int,
    n_bins: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    counts, edges = np.histogram(fid_values, bins=bins, density=False)
    p_emp = counts.astype(np.float64)
    if p_emp.sum() == 0:
        p_emp = np.ones_like(p_emp) / len(p_emp)
    else:
        p_emp /= p_emp.sum()

    mids = 0.5 * (edges[:-1] + edges[1:])
    width = edges[1] - edges[0]
    p_haar = haar_pdf_fidelity(mids, dim=dim) * width
    p_haar /= p_haar.sum()
    return edges, mids, p_emp, p_haar


def kl_divergence(p: np.ndarray, q: np.ndarray, eps: float = 1e-12) -> float:
    p_s = p + eps
    q_s = q + eps
    p_s /= p_s.sum()
    q_s /= q_s.sum()
    return float(np.sum(p_s * np.log(p_s / q_s)))


# ---------------------------------------------------------------------------
# Expressibility sampling on hardware
# ---------------------------------------------------------------------------


def bind_ansatz(
    ansatz_fn: Callable[[int, int], QuantumCircuit],
    n_qubits: int,
    depth: int,
    theta_values: np.ndarray,
) -> QuantumCircuit:
    qc = ansatz_fn(n_qubits, depth)
    ordered_params = list(qc.parameters)
    bind_map = {p: float(v) for p, v in zip(ordered_params, theta_values)}
    return qc.assign_parameters(bind_map, inplace=False)


def sample_hardware_fidelities(
    backend,
    ansatz_fn: Callable[[int, int], QuantumCircuit],
    n_qubits: int,
    depth: int,
    n_samples: int,
    seed: int,
    shots: int,
    optimization_level: int,
    seed_transpiler: int | None,
    max_circuits_per_job: int,
    ansatz_label: str = "",
) -> list[dict[str, object]]:
    qc_template = ansatz_fn(n_qubits, depth)
    n_params = len(qc_template.parameters)
    rng = np.random.default_rng(seed)
    rows: list[dict[str, object]] = []

    for sample_index in range(n_samples):
        theta_a = rng.uniform(0.0, 2.0 * np.pi, n_params)
        theta_b = rng.uniform(0.0, 2.0 * np.pi, n_params)
        bound_a = bind_ansatz(ansatz_fn, n_qubits, depth, theta_a)
        bound_b = bind_ansatz(ansatz_fn, n_qubits, depth, theta_b)

        print(
            f"\n  sample {sample_index + 1}/{n_samples} "
            f"({ansatz_label}, depth={depth})"
        )

        rho_a_lin, rho_a_phys, diag_a = tomography_density_matrices(
            bound_a,
            backend,
            n_qubits=n_qubits,
            shots=shots,
            optimization_level=optimization_level,
            seed_transpiler=seed_transpiler,
            max_circuits_per_job=max_circuits_per_job,
            label=f"{ansatz_label} state A",
        )
        rho_b_lin, rho_b_phys, diag_b = tomography_density_matrices(
            bound_b,
            backend,
            n_qubits=n_qubits,
            shots=shots,
            optimization_level=optimization_level,
            seed_transpiler=seed_transpiler,
            max_circuits_per_job=max_circuits_per_job,
            label=f"{ansatz_label} state B",
        )

        f_lin = hardware_overlap(rho_a_lin, rho_b_lin)
        f_phys = hardware_overlap(rho_a_phys, rho_b_phys)

        print(
            f"    F (linear inv.)    = {f_lin:.4f}\n"
            f"    F (physical proj.) = {f_phys:.4f}"
        )

        rows.append(
            {
                "sample_index": sample_index,
                "fidelity_linear": f_lin,
                "fidelity_physical": f_phys,
                "trace_a_linear": diag_a["trace_linear"],
                "trace_a_physical": diag_a["trace_physical"],
                "trace_b_linear": diag_b["trace_linear"],
                "trace_b_physical": diag_b["trace_physical"],
                "purity_a_linear": diag_a["purity_linear"],
                "purity_a_physical": diag_a["purity_physical"],
                "purity_b_linear": diag_b["purity_linear"],
                "purity_b_physical": diag_b["purity_physical"],
            }
        )

    return rows


def compute_kl_for_fidelities(
    fidelities: np.ndarray,
    dim: int,
    n_bins: int,
    eps: float,
) -> tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    _, mids, p_emp, p_haar = binned_distributions(fidelities, dim, n_bins)
    kl = kl_divergence(p_emp, p_haar, eps)
    return kl, mids, p_emp, p_haar


def circuits_per_fidelity_sample(n_qubits: int) -> int:
    return 2 * (3 ** n_qubits)


def total_circuits(
    n_ansatzes: int,
    n_depths: int,
    n_samples: int,
    n_qubits: int,
) -> int:
    return n_ansatzes * n_depths * n_samples * circuits_per_fidelity_sample(n_qubits)


def estimate_wall_time_minutes(
    n_ansatzes: int,
    n_depths: int,
    n_samples: int,
    minutes_per_state: float = MINUTES_PER_STATE_TOMOGRAPHY,
) -> float:
    """Each fidelity pair needs two full 3^n tomography sweeps (state A and B)."""
    n_pairs = n_ansatzes * n_depths * n_samples
    return n_pairs * 2.0 * minutes_per_state


# ---------------------------------------------------------------------------
# Self-check (no hardware)
# ---------------------------------------------------------------------------


def verify_haar_kl_helpers() -> None:
    dim = 2 ** NUM_QUBITS
    rng = np.random.default_rng(0)
    haar_samples = 1.0 - rng.random(50_000) ** (1.0 / (dim - 1))
    kl, _, _, _ = compute_kl_for_fidelities(haar_samples, dim, N_BINS, EPS)
    assert kl < 0.05, f"Haar self-samples should have low KL, got {kl}"


def verify_expectation_endianness() -> None:
    counts_all_zero = {"00000": 1000}
    assert abs(expectation_from_counts(counts_all_zero, "ZZZZZ") - 1.0) < 1e-12

    # Qiskit bitstrings are little-endian: rightmost char is qubit 0.
    counts_q0_one = {"00001": 1000}
    assert abs(expectation_from_counts(counts_q0_one, "ZZZZZ") - (-1.0)) < 1e-12
    assert abs(expectation_from_counts(counts_q0_one, "IZZZI") - 1.0) < 1e-12


def verify_projection_psd() -> None:
    dim = 8
    rng = np.random.default_rng(1)
    psi = rng.normal(size=dim) + 1j * rng.normal(size=dim)
    psi /= np.linalg.norm(psi)
    rho = np.outer(psi, psi.conj())
    rho_noisy = rho + 0.01 * (rng.normal(size=(dim, dim)) + 1j * rng.normal(size=(dim, dim)))
    rho_noisy = 0.5 * (rho_noisy + rho_noisy.conj().T)
    rho_phys = project_to_physical(rho_noisy)
    eigvals = np.linalg.eigvalsh(rho_phys)
    assert np.all(eigvals >= -1e-10)
    trace = float(np.real(np.trace(rho_phys)))
    assert 0.0 < trace <= 1.0 + 1e-8


def verify_statevector_vs_tomography_overlap() -> None:
    from qiskit.quantum_info import Statevector

    n_qubits = 2
    qc_a = QuantumCircuit(n_qubits)
    qc_a.h(0)
    qc_a.cx(0, 1)
    qc_b = QuantumCircuit(n_qubits)
    qc_b.ry(0.7, 0)
    qc_b.rz(1.1, 1)

    psi_a = Statevector.from_instruction(qc_a).data
    psi_b = Statevector.from_instruction(qc_b).data
    ideal = abs(np.vdot(psi_a, psi_b)) ** 2

    rho_b = np.outer(psi_b, psi_b.conj())
    simulated_counts: dict[tuple[str, ...], dict[str, int]] = {}
    for basis_tuple in all_basis_settings(n_qubits):
        qc = qc_b.copy()
        tomo = add_tomography_rotations(qc, basis_tuple)
        sv = Statevector.from_instruction(tomo.remove_final_measurements(inplace=False))
        probs = sv.probabilities_dict()
        counts = {k.replace(" ", ""): int(round(v * 10_000)) for k, v in probs.items()}
        simulated_counts[basis_tuple] = counts

    rho_lin = reconstruct_rho(simulated_counts, n_qubits)
    rho_phys = project_to_physical(rho_lin)
    rho_a = np.outer(psi_a, psi_a.conj())
    tomography_overlap = hardware_overlap(rho_a, rho_phys)
    assert abs(tomography_overlap - ideal) < 0.05, (
        f"tomography overlap {tomography_overlap} vs ideal {ideal}"
    )


def run_self_check() -> None:
    verify_haar_kl_helpers()
    verify_expectation_endianness()
    verify_projection_psd()
    verify_statevector_vs_tomography_overlap()
    print("All self-checks passed.")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------




## 4. Self-check (no hardware)

In [ ]:
run_self_check()

## 5. Connect to IQM Spark

Set `IQM_TOKEN` in your environment or enter the token when prompted.


In [ ]:
iqm_backend = connect_to_iqm_backend(IQM_URL)
print(f"Connected to backend: {iqm_backend}  (n_qubits = {iqm_backend.num_qubits})")


## 6. Hardware sweep

In [ ]:
stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = NOTEBOOK_DIR / f"iqm_kl_expressibility_{stamp}"
output_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
fidelity_rows = []
t_start = time.perf_counter()

for depth in DEPTHS:
    for ansatz_name, ansatz_fn in ANSATZES.items():
        depth_seed = SEED + 100 * depth + (1 if ansatz_name == "ansatz_simulator" else 0)
        print(f"\n=== {ansatz_name} depth={depth} (seed={depth_seed}) ===")

        sample_rows = sample_hardware_fidelities(
            iqm_backend,
            ansatz_fn,
            n_qubits=NUM_QUBITS,
            depth=depth,
            n_samples=N_SAMPLES,
            seed=depth_seed,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=None,
            max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
            ansatz_label=ansatz_name,
        )

        f_phys = np.array([r["fidelity_physical"] for r in sample_rows])
        f_lin = np.array([r["fidelity_linear"] for r in sample_rows])
        kl_phys, _, _, _ = compute_kl_for_fidelities(f_phys, DIM, N_BINS, EPS)
        kl_lin, _, _, _ = compute_kl_for_fidelities(f_lin, DIM, N_BINS, EPS)

        summary_rows.append(
            {
                "ansatz": ansatz_name,
                "depth": depth,
                "n_qubits": NUM_QUBITS,
                "n_samples": N_SAMPLES,
                "shots": SHOTS,
                "n_bins": N_BINS,
                "eps": EPS,
                "seed": depth_seed,
                "kl_physical": kl_phys,
                "kl_linear": kl_lin,
                "f_physical_mean": float(np.mean(f_phys)),
                "f_physical_std": float(np.std(f_phys)),
                "f_linear_mean": float(np.mean(f_lin)),
                "f_linear_std": float(np.std(f_lin)),
            }
        )
        for row in sample_rows:
            fidelity_rows.append({"ansatz": ansatz_name, "depth": depth, **row})

wall_min = (time.perf_counter() - t_start) / 60.0

results_df = pd.DataFrame(summary_rows)
fidelities_df = pd.DataFrame(fidelity_rows)
results_path = output_dir / "iqm_kl_results.csv"
fidelities_path = output_dir / "iqm_kl_fidelities.csv"
manifest_path = output_dir / "run_manifest.json"
results_df.to_csv(results_path, index=False)
fidelities_df.to_csv(fidelities_path, index=False)

manifest = {
    "created_utc": datetime.now(tz=timezone.utc).isoformat(),
    "backend": str(iqm_backend),
    "iqm_url": IQM_URL,
    "source_notebook": "evaluation_and_comparison/iqm_kl_expressibility.ipynb",
    "tomography_source": "full_odra_fidelity.ipynb",
    "method": "hardware_hardware_overlap_tomography",
    "fidelity_definition": "Tr(rho_a @ rho_b) from full 3^n Pauli tomography",
    "kl_direction": "P_hardware || P_Haar",
    "n_qubits": NUM_QUBITS,
    "dim": DIM,
    "depths": list(DEPTHS),
    "ansatzes": list(ANSATZES.keys()),
    "n_samples": N_SAMPLES,
    "shots": SHOTS,
    "n_bins": N_BINS,
    "eps": EPS,
    "seed": SEED,
    "optimization_level": OPTIMIZATION_LEVEL,
    "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    "circuits_per_fidelity_sample": circuits_per_pair,
    "total_tomography_circuits": total_circuits,
    "wall_time_minutes": wall_min,
    "outputs": [results_path.name, fidelities_path.name],
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print(f"\nSaved outputs to {output_dir}")
print(f"Total wall time: {wall_min:.1f} min")
results_df


## 7. Comparison table

In [ ]:
print("KL comparison (lower is better, physical projection):")
print("depth | ansatz           | KL_physical | F_phys_mean")
print("-" * 55)
for _, row in results_df.iterrows():
    print(
        f"{int(row['depth']):>5} | {row['ansatz']:<16} | "
        f"{row['kl_physical']:.6f}    | {row['f_physical_mean']:.4f}"
    )


## 8. Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ansatz_name in ANSATZES:
    sub = results_df[results_df["ansatz"] == ansatz_name].sort_values("depth")
    ax.plot(sub["depth"], sub["kl_physical"], marker="o", label=ansatz_name)
ax.set_xlabel("Depth")
ax.set_ylabel("KL(P_hardware || P_Haar)")
ax.set_title("KL vs depth on IQM Spark (lower is better)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
PLOT_DEPTH = DEPTHS[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, ansatz_name in zip(axes, ANSATZES):
    sub = fidelities_df[
        (fidelities_df["ansatz"] == ansatz_name) & (fidelities_df["depth"] == PLOT_DEPTH)
    ]
    f_vals = sub["fidelity_physical"].to_numpy()
    _, mids, p_emp, p_haar = binned_distributions(f_vals, DIM, N_BINS)
    width = 1.0 / N_BINS
    ax.bar(mids, p_emp / width, width=width, alpha=0.5, label="hardware empirical")
    haar_curve = haar_pdf_fidelity(mids, DIM)
    ax.plot(mids, haar_curve, color="black", lw=2, label="Haar density")
    ax.set_title(f"{ansatz_name} depth={PLOT_DEPTH}")
    ax.set_xlabel("F = Tr(rho_a rho_b)")
    ax.set_ylabel("density")
    ax.legend()

plt.suptitle("Hardware overlap histogram vs Haar (physical projection)")
plt.tight_layout()
plt.show()
